In [2]:
!pip install numpy==2.2.3 --quiet
!pip install scikit-learn==1.6.0 --quiet
from sklearn.linear_model import Lasso
import numpy as np

In [15]:
GLOBAL_SEED = 7
def barron_proxy(f, d, N=25000, base_M=60,
                 scales_mult=None, alpha=None, seed=GLOBAL_SEED):
    scales = [0.3, 1.0, 3.0, 5.0, 10.0, 50.0]
    alpha = 1e-6 if d <= 5 else 1e-5

    np.random.seed(seed)
    X = np.random.rand(N, d).astype(np.float32)
    y = f(X).astype(np.float32)
    y = y - np.mean(y)

    all_omegas = []
    all_Z = []

    for s in scales:
        M = int(base_M * d)
        omega = np.random.randn(M, d) * s
        b = np.random.uniform(0, 2 * np.pi, M).astype(np.float32)

        Z = np.sqrt(2.0 / M) * np.cos(X @ omega.T + b)
        all_omegas.append(omega)
        all_Z.append(Z)

    Z_all = np.hstack(all_Z)
    omega_all = np.vstack(all_omegas)

    lasso = Lasso(
        alpha=alpha,
        max_iter=7000,
        tol=1e-4,
        fit_intercept=False,
        random_state=seed
    )
    lasso.fit(Z_all, y)
    a = lasso.coef_

    return np.sum(np.abs(a) * np.linalg.norm(omega_all, axis=1))
def f_radial(X):
    return np.exp(-2 * np.sum((X-0.5)**2, axis=1))

def f_ridge_sum(X):
    d = X.shape[1]
    a1 = np.array([1,1] + [0.5]*(d-2))[:d]
    a2 = np.array([0.5,1,1] + [0.5]*(d-3))[:d]
    a3 = np.array([1,0.5,1,1] + [1]*(d-4))[:d]
    b1,b2,b3 = 0.1,0.7,0.3
    return np.tanh(X @ a1 + b1) + np.sin(X @ a2 + b2) + np.cos(X @ a3 + b3)

def f_fourier_narrow(X):
    d = X.shape[1]
    w1 = np.ones(d)
    w2 = np.array([2,1.5] + [1]*(d-2))[:d]
    return np.cos(X @ w1) + 0.5*np.sin(X @ w1) + 0.8*np.cos(X @ w2) - 0.3*np.sin(X @ w2)

def f_fourier_noisy(X):
    d = X.shape[1]
    w1 = np.ones(d)
    w2 = np.ones(d)*10
    return np.cos(X @ w1) + 0.5*np.sin(X @ w1) + 0.8*np.cos(X @ w2) - 0.3*np.sin(X @ w2)

names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]
funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]
ds = [2, 5,10, 20]

print("Proxy-норма Баррона:\n")
for d in ds:
    print(f"d = {d}")
    for f, name in zip(funcs, names):
        a = 1e-5
        proxy = barron_proxy(f, d, N=1000*d, alpha=a)
        print(f"{name:18} : proxy-norm ≈ {proxy:.3f}")

Proxy-норма Баррона:

d = 2
Radial             : proxy-norm ≈ 27.034
Ridge sum          : proxy-norm ≈ 31.144
Fourier narrow     : proxy-norm ≈ 33.426


/opt/anaconda3/envs/Downloads/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.696e-01, tolerance: 8.383e-02
  model = cd_fast.enet_coordinate_descent(


Fourier noisy      : proxy-norm ≈ 1816.413
d = 5


/opt/anaconda3/envs/Downloads/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.956e-02, tolerance: 1.055e-02
  model = cd_fast.enet_coordinate_descent(


Radial             : proxy-norm ≈ 94.679
Ridge sum          : proxy-norm ≈ 127.188


...

/opt/anaconda3/envs/Downloads/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.209e-01, tolerance: 9.264e-03
  model = cd_fast.enet_coordinate_descent(


Radial             : proxy-norm ≈ 146.853


/opt/anaconda3/envs/Downloads/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.026e+00, tolerance: 9.623e-02
  model = cd_fast.enet_coordinate_descent(


Ridge sum          : proxy-norm ≈ 782.740


KeyboardInterrupt: 